# Reranker: trait-fit classifier (Phase 5)

Fine-tunes DistilBERT on `"[traits] [SEP] [poem]"` to score whether a quatrain roasts the given traits. The reranker
adds the score as `w5·trait_fit`, for its top 5 candidates only. Each trait is written out with its synonyms and flip
side from `traits.json` (`confident: self-assured, bold, sure of myself, cocky, arrogant, all talk`), so a held-out
trait the model never saw in training still comes with words it knows.

**Data** (`reranker/trait_fit.py`, built on the fly, the same for the same seed):
- **Positives:** the disses (`data/disses/corpus/`) with their own traits. The generic fallback disses have no traits
  and are left out.
- **Negatives:** the 1e set (`data/negatives/corpus/negatives.jsonl`) plus disses with shuffled lines.
  - `flat`, `non_rhyming`, `off_topic`: split 80/10/10 within each type.
  - `wrong_trait`: 1e built these from train disses, so they stay in train. Val and test get their own from their own
    disses, with the same rule (never a trait from the label's `CLOSE` group) and at the train ratio.
  - `shuffled`: disses with their lines reordered so that neither couplet stays together.
- **Held-out traits** (test only): the 146 held-out disses as positives, each also labelled with a training trait, and
  as many training-trait disses labelled with a held-out trait. So held-out traits are tested both as the label and
  as the subject of the poem.

The test set is reported for training-trait and held-out-trait rows separately: AUC, accuracy, precision, recall and
F1 at 0.5. For each negative type it also gives how often that type is scored as a fit. The same code runs locally
with `python reranker/train_trait_fit.py`.

**On Colab:**
1. On your machine, from the project root, run `python reranker/make_colab_zip.py`. It writes `reranker_colab.zip`
   (about 1 MB): the `reranker/` code, `utils.py`, `config.yaml`, `traits.json`, the diss corpus, the negatives and
   the two files with the trait groups.
2. Runtime → Change runtime type → **T4 GPU**, then Run all. The first cell asks you to upload `reranker_colab.zip`. To
   skip the prompt, drag it into the Files panel (`/content/`) or put it in `MyDrive/CS425/` beforehand.

The model is saved to Google Drive (`MyDrive/CS425/checkpoints/trait_fit_best/`) and the results to
`MyDrive/CS425/checkpoints/trait_fit_results/`. The results are also downloaded as `trait_fit_results.zip` at the end
of the Test section; unzip that into `eval/results/trait_fit/`. After changing the code or the data, make a new zip
and upload it again.

**Expected time on a T4:** 1–2 min for setup and downloads, then the training sweep: 6 runs of about 1.5–2.5 min each
(fp16), each evaluated on test. **About 10–15 min in all.**

The last section scores poems of your own.

In [ ]:
import os, sys, shutil, subprocess

ZIP_NAME = "reranker_colab.zip"                      # made by python reranker/make_colab_zip.py
DRIVE_ZIP = f"/content/drive/MyDrive/CS425/{ZIP_NAME}"
DRIVE_CKPT_DIR = "/content/drive/MyDrive/CS425/checkpoints"
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"     # a few batches only, to check the notebook runs end to end

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive, files
    drive.mount("/content/drive")
    root = "/content/CS425-Project"
    os.chdir("/content")
    zip_path = next((p for p in (f"/content/{ZIP_NAME}", DRIVE_ZIP) if os.path.exists(p)), None)
    if zip_path is None:
        print(f"upload {ZIP_NAME} (python reranker/make_colab_zip.py makes it)")
        zip_path = os.path.join("/content", next(iter(files.upload())))
    shutil.rmtree(root, ignore_errors=True)           # a fresh copy, so an older upload never mixes in
    shutil.unpack_archive(zip_path, root)
    print("unpacked", zip_path, "->", root)
    # torch, transformers and pyyaml come with Colab; the trait groups import form_checks, which needs these two
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pronouncing", "wordfreq"], check=True)
    os.chdir(root)
    CKPT_DIR = DRIVE_CKPT_DIR
else:
    if os.path.basename(os.getcwd()) == "reranker":
        os.chdir("..")
    CKPT_DIR = "checkpoints"
if SMOKE_TEST:
    CKPT_DIR = os.path.join(CKPT_DIR, "smoke_test")
sys.path.insert(0, os.getcwd())
print("project:", os.getcwd(), "| checkpoints:", CKPT_DIR, "| smoke test:", SMOKE_TEST)

In [ ]:
import json, random
import torch
from utils import load_config, project_path, set_seed
from reranker.trait_fit import TraitFit, encode, load, poem_text, read_jsonl, trait_text
from reranker.train_trait_fit import evaluate, load_rows, split_counts, sweep, sweep_table

cfg = load_config()
hp = dict(cfg["trait_fit"])                  # hyperparameters from config.yaml
SEED = set_seed(cfg["seed"])
device = "cuda" if torch.cuda.is_available() else "cpu"
if IN_COLAB and device != "cuda":
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, or training takes about 10x longer")
CKPT = os.path.join(CKPT_DIR, "trait_fit_best")
RESULTS_DIR = project_path("eval/results/trait_fit")
if SMOKE_TEST:                               # keep the real results
    RESULTS_DIR = project_path(CKPT_DIR) / "trait_fit_results"

rows = load_rows(cfg)                        # {"train", "val", "test": [row, ...], "info": traits.json}
info = rows["info"]
print("device:", device, torch.cuda.get_device_name() if device == "cuda" else "", "| model:", hp["model"])
for s, c in split_counts({s: rows[s] for s in ("train", "val", "test")}).items():
    n = sum(v for k, v in c.items() if k != "heldout")
    print(f"{s:5s} {n:5d}  " + ", ".join(f"{k} {v}" for k, v in sorted(c.items())))

In [ ]:
# One test row of each type, as the model sees it: trait descriptions [SEP] the poem, a slash at each line break.
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(hp["model"])
for t in ["positive", "wrong_trait", "flat", "non_rhyming", "off_topic", "shuffled"]:
    r = next(r for r in rows["test"] if r["type"] == t and not r["heldout"])
    n = len(encode(tok, [r], info, hp["describe_traits"], hp["max_len"])[0])
    about = f" (written for {r['about']})" if r["about"] else ""
    print(f"[{t}] label {r['label']}, {n} pieces{about}\n  {trait_text(r['traits'], info, hp['describe_traits'])}"
          f"\n  [SEP] {poem_text(r['lines'])}\n")

## Training: learning rate × seed sweep
This trains one model for each learning rate in `LRS` and seed in `SEEDS` (by default 2e-5 and 5e-5 × 3 seeds). Each
run trains for up to `EPOCHS` epochs, stops early after 2 epochs without improvement, and keeps the epoch with the
best val AUC. AUC is used because the reranker only uses the order of the scores. The learning rate with the best mean
val AUC wins, and its run with the best val AUC becomes the model. **Test results are only reported, never used to
choose.**

Runs are saved to the Colab disk (`/content/trait_fit_sweep/`). Only the chosen run is copied to Drive, where it
replaces `trait_fit_best/`. Each run takes about 1.5–2.5 min on a T4, so the default 6 take **10–15 min**. For a single
run, set `LRS = [hp["lr"]]` and `SEEDS = [SEED]`.

In [ ]:
LRS = [2e-5, 5e-5]
SEEDS = [42, 43, 44]
EPOCHS = hp["epochs"]          # early stopping often ends a run sooner
MAX_BATCHES = None
SWEEP_DIR = "/content/trait_fit_sweep" if IN_COLAB else os.path.join(CKPT_DIR, "trait_fit_sweep")
if SMOKE_TEST:
    rows = {**{s: rows[s][:64] for s in ("train", "val")}, "test": rows["test"][:300], "info": info}
    LRS, SEEDS, EPOCHS, MAX_BATCHES = [3e-5], [42, 43], 1, 3

runs, chosen = sweep(rows, hp, LRS, SEEDS, EPOCHS, device, SWEEP_DIR, max_batches=MAX_BATCHES)
if device == "cuda":
    print(f"peak GPU memory {torch.cuda.max_memory_allocated() / 2**30:.1f} GB")

In [ ]:
sweep_table(runs)
print(f"\nchosen: {chosen['name']} (lr with the best mean val AUC, then its seed with the best val AUC)")
history = chosen["history"]
print(f"{'epoch':>5} {'train loss':>10} {'val AUC':>8} {'val acc':>8} {'val F1':>7} {'s':>5}")
for h in history:
    print(f"{h['epoch']:5d} {h['train_loss']:10.4f} {h['val_auc']:8.4f} {h['val_accuracy']:8.3f} {h['val_f1']:7.3f} "
          f"{h['seconds']:5.0f}")
shutil.rmtree(CKPT, ignore_errors=True)
shutil.copytree(chosen["ckpt"], CKPT, ignore=shutil.ignore_patterns("results"))
print("copied", chosen["ckpt"], "->", CKPT)

## Test
This evaluates the chosen run on the test set. The held-out-trait rows were never seen in training. It writes
`summary.json`, `test_errors.jsonl`, `history.json` and `sweep.json` (every run of the sweep) to
`eval/results/trait_fit/`, and the cells below copy them to Drive and download them.

- **scored fit**, per type: the share of rows scored ≥ 0.5. For `positive` that is recall; for a negative type it is
  how often that kind of bad candidate would get through (lower is better).
- `wrong_trait` is the hardest type: the poem is a real diss and only the label is wrong. In the held-out group, half
  of these have a held-out label and half a held-out poem.
- **Time to score 5 candidates** on the GPU and on the CPU. The demo runs the reranker on a laptop CPU.

In [ ]:
model, tok, meta = load(CKPT, device)
summary = evaluate(model, tok, meta, rows["test"], hp, device, RESULTS_DIR)
shutil.copy(os.path.join(CKPT, "history.json"), RESULTS_DIR / "history.json")
json.dump({"lrs": LRS, "seeds": SEEDS, "epochs": EPOCHS, "chosen": chosen["name"],
           "split_counts": split_counts({s: rows[s] for s in ("train", "val", "test")}),
           "runs": [{k: v for k, v in r.items() if k != "ckpt"} for r in runs]},
          open(RESULTS_DIR / "sweep.json", "w"), indent=2)

In [ ]:
# Copy the results to Drive and download them as trait_fit_results.zip; unzip it into eval/results/trait_fit/ in the repo.
if IN_COLAB:
    dest = os.path.join(CKPT_DIR, "trait_fit_results")
    shutil.copytree(RESULTS_DIR, dest, dirs_exist_ok=True)
    print("copied", sorted(os.listdir(dest)), "to", dest)
    files.download(shutil.make_archive("/content/trait_fit_results", "zip", RESULTS_DIR))

## Errors
The test rows on the wrong side of 0.5, worst first, up to 3 per type.

In [ ]:
errors = read_jsonl(RESULTS_DIR / "test_errors.jsonl")
print(len(errors), "of", len(rows["test"]), "test rows on the wrong side of 0.5")
for t in sorted({e["type"] for e in errors}):
    sel = [e for e in errors if e["type"] == t]
    print(f"\n== {t}: {len(sel)}")
    for e in sel[:3]:
        about = f" | written for {e['about']}" if e["about"] else ""
        held = " | held-out" if e["heldout"] else ""
        print(f"- score {e['score']:.3f} | traits {e['traits']}{about}{held}\n    {e['poem'][:200]}")

## Try it
`TraitFit.score(traits, poems)` is what the reranker calls: one score in [0, 1] for each candidate poem (its four
lines, or a single string with one line per row).

In [ ]:
fit = TraitFit(model, tok, meta, device)
d = next(r for r in rows["test"] if r["type"] == "positive" and r["heldout"])     # a diss for a held-out trait
order = [0, 2, 1, 3]
print("diss written for", d["traits"], "(held out):")
for label, traits, lines in [("its own traits", d["traits"], d["lines"]),
                             ("a wrong trait", ["athletic"], d["lines"]),
                             ("lines shuffled", d["traits"], [d["lines"][i] for i in order])]:
    print(f"  {label:15s} {traits} -> {fit.score(traits, [lines])[0]:.3f}")

In [ ]:
poems = [
    "you say you're always early, first one through the door,\n"
    "but you're just there to count the minutes we were late before.\n"
    "you claim your schedule's perfect, every second planned,\n"
    "yet your calendar's the only friend that'll hold your hand.",
    "you wake up at the crack of dawn and greet the morning light,\n"
    "you plan your day with careful notes and always get it right.\n"
    "you never miss a meeting and you're never in a rush,\n"
    "your friends can always count on you, without a fuss or hush.",
    "the kettle starts to whistle as the morning settles in,\n"
    "the toast is getting crispy and the radio is thin.\n"
    "the cat is on the windowsill, the sun is on the floor,\n"
    "and someone's left the mail outside the neighbour's door.",
]
for traits in (["punctual"], ["organized"], ["funny"]):
    print(traits, [round(s, 3) for s in fit.score(traits, poems)], "(roast, flat praise, off topic)")